# Frozen WavLM LOSO Training

Notebook này giả định bạn đang mở trực tiếp trong repo `conversational_ser` đã có sẵn code và data.

Mục tiêu: chứng minh idea CIM bằng setup kiểm soát biến tốt hơn:

- WavLM frozen
- fixed mean-pooled acoustic embeddings
- LOSO trên 5 session
- so sánh `Baseline`, `CDM`, `CIM`, `CIM-zero`, `CIM-shuffled`

Notebook chỉ chạy config có sẵn trong `configs/`, không sinh YAML tạm. Nếu cần đổi epoch, learning rate, batch size, W&B, output dir hoặc session list thì sửa trực tiếp file YAML.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from datetime import datetime

import torch

PROJECT_DIR = Path.cwd().resolve()
assert (PROJECT_DIR / 'scripts').is_dir(), f'PROJECT_DIR không phải repo root: {PROJECT_DIR}'
assert (PROJECT_DIR / 'configs').is_dir(), f'Thiếu configs/: {PROJECT_DIR}'

print('PROJECT_DIR =', PROJECT_DIR)
print('CUDA        =', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## Optional Login

Giữ kiểu login trực tiếp nếu chạy trên Vast.ai/Kaggle. Bỏ comment và điền token khi cần.


In [ ]:
# !wandb login YOUR_WANDB_KEY
# !huggingface-cli login --token YOUR_HF_TOKEN


## Configs

Các config mặc định dùng `precompute.enabled=true`, `model.freeze_wavlm=true`, `model.unfreeze_last_n_layers=0`. Đây là setup chính để tách tác dụng của temporal modeling khỏi tác dụng fine-tune SSL backbone.


In [ ]:
MODEL_SPECS = {
    'baseline': {
        'config': PROJECT_DIR / 'configs' / 'wavlm_baseline_loso.yaml',
        'trainer': 'scripts.train_wavlm_baseline',
        'output_dir': PROJECT_DIR / 'results' / 'wavlm_baseline_loso',
    },
    'cdm': {
        'config': PROJECT_DIR / 'configs' / 'wavlm_cdm_loso.yaml',
        'trainer': 'scripts.train_wavlm_cdm',
        'output_dir': PROJECT_DIR / 'results' / 'wavlm_cdm_loso',
    },
    'cim': {
        'config': PROJECT_DIR / 'configs' / 'wavlm_cim_loso.yaml',
        'trainer': 'scripts.train_wavlm_cim',
        'output_dir': PROJECT_DIR / 'results' / 'wavlm_cim_loso',
    },
    'cim_zero': {
        'config': PROJECT_DIR / 'configs' / 'wavlm_cim_zero_loso.yaml',
        'trainer': 'scripts.train_wavlm_cim',
        'output_dir': PROJECT_DIR / 'results' / 'wavlm_cim_zero_loso',
    },
    'cim_shuffled': {
        'config': PROJECT_DIR / 'configs' / 'wavlm_cim_shuffled_loso.yaml',
        'trainer': 'scripts.train_wavlm_cim',
        'output_dir': PROJECT_DIR / 'results' / 'wavlm_cim_shuffled_loso',
    },
}

for name, spec in MODEL_SPECS.items():
    assert spec['config'].is_file(), f'Thiếu config cho {name}: {spec["config"]}'
    print(name, '->', spec['config'])


Không có bước sinh config. Các trainer sẽ nhận trực tiếp file YAML trong `configs/`.


## Commands

Copy từng command để chạy riêng trong terminal, hoặc dùng cell tiếp theo để chạy tuần tự.


In [ ]:
COMMANDS = []
for name, spec in MODEL_SPECS.items():
    command = [sys.executable, '-u', '-m', spec['trainer'], '--config', str(spec['config'])]
    COMMANDS.append((name, command))
    print(f'[{name}]')
    print(' '.join(command))


In [ ]:
# Chạy một model cụ thể: baseline, cdm, cim, cim_zero, cim_shuffled.
# Để trống nếu chỉ muốn in command.
RUN_MODEL = os.environ.get('SER_RUN_MODEL', '')

if RUN_MODEL:
    selected = [(name, cmd) for name, cmd in COMMANDS if name == RUN_MODEL]
    assert selected, f'RUN_MODEL không hợp lệ: {RUN_MODEL}'
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    for name, command in selected:
        print(f'[{datetime.now():%H:%M:%S}] Running {name}:', ' '.join(command))
        process = subprocess.Popen(
            command,
            cwd=PROJECT_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
        return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, command)
else:
    print('SER_RUN_MODEL chưa set, notebook chỉ in command.')


## Upload Hugging Face

Sau khi train xong, cell này upload artifact của model đã chọn hoặc toàn bộ model trong `MODEL_SPECS` lên Hugging Face. Không tự chạy nếu chưa bật `SER_UPLOAD_TO_HF=1`.


In [ ]:
HF_REPO_ID = os.environ.get('HF_REPO_ID', 'ngocbao05/ser')
HF_REPO_TYPE = os.environ.get('HF_REPO_TYPE', 'model')
HF_BASE_PATH = os.environ.get('HF_BASE_PATH', 'frozen_wavlm_loso')
UPLOAD_MODEL = os.environ.get('SER_UPLOAD_MODEL', os.environ.get('SER_RUN_MODEL', '')).strip()
UPLOAD_TO_HF = os.environ.get('SER_UPLOAD_TO_HF', '0') == '1'


def latest_artifact_dir(output_dir: Path) -> Path:
    run_dirs = sorted((output_dir / 'cross_session').glob('run_*'))
    if run_dirs:
        return run_dirs[-1]
    return output_dir


upload_commands = []
for name, spec in MODEL_SPECS.items():
    if UPLOAD_MODEL and name != UPLOAD_MODEL:
        continue
    artifact_dir = latest_artifact_dir(spec['output_dir'])
    if not artifact_dir.exists():
        print(f'[{name}] skip, artifact not found:', artifact_dir)
        continue
    path_in_repo = f'{HF_BASE_PATH}/{name}/{artifact_dir.name}'
    command = [
        'hf', 'upload', HF_REPO_ID, str(artifact_dir), path_in_repo,
        '--repo-type', HF_REPO_TYPE,
        '--exclude', '**/wandb/**',
        '--commit-message', f'Upload {name} WavLM last-4 artifacts',
    ]
    upload_commands.append((name, command))
    print(f'[{name}]')
    print(' '.join(command))

if not upload_commands:
    print('Không có artifact nào để upload. Hãy train xong trước hoặc kiểm tra output_dir trong config.')


In [ ]:
if UPLOAD_TO_HF:
    for name, command in upload_commands:
        print(f'[{datetime.now():%H:%M:%S}] Uploading {name}:', ' '.join(command))
        subprocess.run(command, cwd=PROJECT_DIR, check=True)
else:
    print('SER_UPLOAD_TO_HF=0, chỉ in command upload. Set SER_UPLOAD_TO_HF=1 để upload thật.')
